# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [1]:
# 프롬프트 파일 읽기
with open("../make_tool_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**TODO: 팀에서 선택한 도메인과 필요한 도구를 작성하세요**

```
팀 도메인: [증권 도메인]

필요한 도구 목록:
1. get_current_price : (실시간 주가 조회)

입력: 종목 코드 (티커)

출력: 현재가, 전일 대비 등락률, 당일 거래량

책임: 특정 종목의 '현재' 가격 정보만 빠르고 정확하게 가져옵니다. 

2. get_historical_ohlcv : (과거 주가 조회)

입력: 종목 코드, 시작일, 종료일, 기준(일/주/월)

출력: 날짜별 시가(Open), 고가(High), 저가(Low), 종가(Close), 거래량(Volume) 배열

책임: 차트나 백테스팅에 필요한 과거 시계열 데이터를 제공합니다.

3. get_financial_metrics : (핵심 재무지표 조회)

입력: 종목 코드, 연도/분기

출력: PER, PBR, ROE, 매출액, 영업이익

책임: 방대한 재무제표 대신, 투자 판단에 즉시 사용되는 핵심 지표만 요약해 반환합니다.

4. get_company_disclosures : (공시 및 뉴스 검색)

입력: 종목 코드, 날짜 범위, 키워드(선택)

출력: 해당 기업의 주요 공시(실적 발표, 유상증자 등) 및 관련 뉴스 헤드라인 목록

책임: 종목의 가격에 영향을 미칠 수 있는 외부 이벤트 정보를 제공합니다.
```

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. 위의 프롬프트 템플릿을 복사
2. `<이곳에 원하는 Tool 기능을 작성>` 부분에 팀의 도구 요구사항 작성
3. ChatGPT, Claude 등에 입력하여 코드 생성
4. 생성된 코드를 아래 셀에 붙여넣기

**예시 입력:**
```
쇼핑 도메인의 상품 검색 도구를 만들어주세요.

기능:
- 상품명으로 검색
- 가격 범위 필터링
- 카테고리 필터링
- 검색 결과를 JSON 형태로 반환
```

---

## 4. 생성된 도구 코드 테스트

**TODO: AI가 생성한 도구 코드를 아래에 붙여넣으세요**

**중요:** 
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요

In [3]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from typing import List, Dict, Union, Optional
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_current_price(ticker: str) -> dict:
    """특정 종목의 '현재' 가격 정보를 빠르고 정확하게 가져옵니다.

    Args:
        ticker: 종목 코드 (예: 'AAPL' - 애플, '005930.KS' - 삼성전자)

    Returns:
        현재가, 전일 대비 등락률, 당일 거래량을 포함한 딕셔너리
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.fast_info
        
        current_price = info.last_price
        prev_close = info.previous_close
        volume = info.last_volume
        
        change_rate = ((current_price - prev_close) / prev_close) * 100 if prev_close else 0.0
        
        return {
            "ticker": ticker,
            "current_price": round(current_price, 2) if current_price else None,
            "change_rate": round(change_rate, 2) if change_rate else None,
            "volume": int(volume) if volume else 0
        }
    except Exception as e:
        return {"error": f"실패: {str(e)}"}


@tool(parse_docstring=True)
def get_historical_ohlcv(ticker: str, start_date: str, end_date: str, interval: str = '일') -> list:
    """차트나 백테스팅에 필요한 과거 시계열 데이터를 제공합니다.

    Args:
        ticker: 종목 코드 (예: 'AAPL')
        start_date: 시작일 (YYYY-MM-DD 형식)
        end_date: 종료일 (YYYY-MM-DD 형식)
        interval: 데이터 간격 기준 ('일', '주', '월')

    Returns:
        날짜별 시가, 고가, 저가, 종가, 거래량 배열 (리스트 형태)
    """
    interval_map = {'일': '1d', '주': '1wk', '월': '1mo'}
    yf_interval = interval_map.get(interval, '1d')
    
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start=start_date, end=end_date, interval=yf_interval)
        
        if df.empty:
            return []
            
        result = []
        for date, row in df.iterrows():
            result.append({
                "date": date.strftime('%Y-%m-%d'),
                "open": round(row['Open'], 2),
                "high": round(row['High'], 2),
                "low": round(row['Low'], 2),
                "close": round(row['Close'], 2),
                "volume": int(row['Volume'])
            })
            
        return result
    except Exception as e:
        return [{"error": f"실패: {str(e)}"}]


@tool(parse_docstring=True)
def get_financial_metrics(ticker: str, period: str = 'annual') -> dict:
    """투자 판단에 즉시 사용되는 핵심 재무 지표를 요약하여 반환합니다.

    Args:
        ticker: 종목 코드 (예: 'AAPL', '005930.KS')
        period: 'annual'(연간) 또는 'quarterly'(분기)

    Returns:
        매출액, 영업이익, PER, PBR, ROE가 포함된 딕셔너리
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        per = info.get('trailingPE')
        pbr = info.get('priceToBook')
        roe = info.get('returnOnEquity')
        if roe is not None:
            roe = round(roe * 100, 2)
            
        financials = stock.financials if period == 'annual' else stock.quarterly_financials
        
        revenue, operating_income, target_date = None, None, None
        
        if not financials.empty:
            latest_col = financials.columns[0]
            target_date = latest_col.strftime('%Y-%m-%d')
            
            if 'Total Revenue' in financials.index:
                revenue = financials.loc['Total Revenue', latest_col]
            if 'Operating Income' in financials.index:
                operating_income = financials.loc['Operating Income', latest_col]

        return {
            "ticker": ticker,
            "report_date": target_date,
            "period": period,
            "revenue": int(revenue) if pd.notna(revenue) else None,
            "operating_income": int(operating_income) if pd.notna(operating_income) else None,
            "PER": round(per, 2) if per else None,
            "PBR": round(pbr, 2) if pbr else None,
            "ROE_percent": roe
        }
    except Exception as e:
        return {"error": f"실패: {str(e)}"}


@tool(parse_docstring=True)
def get_company_disclosures(ticker: str, start_date: str = "", end_date: str = "", keyword: str = "") -> list:
    """해당 기업의 주요 뉴스 헤드라인 및 정보를 제공합니다.

    Args:
        ticker: 종목 코드 (예: 'AAPL')
        start_date: 검색 시작일 (YYYY-MM-DD 형식, 선택사항, 빈 문자열 가능)
        end_date: 검색 종료일 (YYYY-MM-DD 형식, 선택사항, 빈 문자열 가능)
        keyword: 기사 제목 필터링을 위한 특정 키워드 (선택사항, 빈 문자열 가능)

    Returns:
        날짜, 제목, 출처, 링크가 포함된 딕셔너리 리스트
    """
    try:
        stock = yf.Ticker(ticker)
        news_items = stock.news
        
        if not news_items:
            return []
            
        filtered_news = []
        start_dt = datetime.strptime(start_date, '%Y-%m-%d') if start_date else None
        end_dt = datetime.strptime(end_date, '%Y-%m-%d') if end_date else None

        for item in news_items:
            pub_time = item.get('providerPublishTime')
            if not pub_time:
                continue
                
            news_date = datetime.fromtimestamp(pub_time)
            title = item.get('title', '')
            
            if start_dt and news_date < start_dt:
                continue
            if end_dt and news_date > end_dt:
                continue
            if keyword and keyword.lower() not in title.lower():
                continue
                
            filtered_news.append({
                "date": news_date.strftime('%Y-%m-%d %H:%M:%S'),
                "title": title,
                "publisher": item.get('publisher', 'Unknown'),
                "link": item.get('link', '')
            })
            
        return filtered_news
    except Exception as e:
        return [{"error": f"실패: {str(e)}"}]

ModuleNotFoundError: No module named 'yfinance'

## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [ ]:
# 생성한 4개의 도구를 리스트로 묶습니다.
tools_list = [
    get_current_price,
    get_historical_ohlcv,
    get_financial_metrics,
    get_company_disclosures
]

# 반복문을 통해 각 도구의 정보를 차례대로 출력합니다.
for tool_function_name in tools_list:
    print("=" * 80)
    print("도구 정보")
    print("=" * 80)
    print(f"도구 이름: {tool_function_name.name}")
    print(f"도구 설명: {tool_function_name.description}")
    print(f"\n입력 스키마:")
    import pprint
    pprint.pprint(tool_function_name.args_schema.schema()) 
    # pprint를 쓰면 스키마 딕셔너리가 훨씬 보기 좋게 출력됩니다.
    print("\n")

## 6. 도구 단독 실행 테스트

**TODO: 다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)

In [ ]:
print("=" * 80)
print("도구 테스트 실행")
print("=" * 80)

print("테스트 1: get_current_price (정상 케이스 - 애플 실시간 주가)")
result1 = get_current_price.invoke({"ticker": "AAPL"})
print(result1)
print()

print("테스트 2: get_historical_ohlcv (정상 케이스 - 과거 주가 배열)")
result2 = get_historical_ohlcv.invoke({
    "ticker": "AAPL", 
    "start_date": "2024-01-01", 
    "end_date": "2024-01-05", 
    "interval": "일"
})
print(result2)
print()

print("테스트 3: get_financial_metrics (정상 케이스 - 연간 핵심 재무지표)")
result3 = get_financial_metrics.invoke({
    "ticker": "AAPL", 
    "period": "annual"
})
print(result3)
print()

print("테스트 4: get_company_disclosures (조건 검색 케이스 - 뉴스/공시)")
result4 = get_company_disclosures.invoke({
    "ticker": "AAPL", 
    "keyword": "Apple"
})
# 뉴스는 결과가 길 수 있으므로 상위 2개만 출력하도록 슬라이싱([:2]) 처리했습니다.
print(result4[:2] if isinstance(result4, list) else result4)
print()

print("테스트 5: 에러 케이스 (존재하지 않는 이상한 종목 코드 입력)")
result5 = get_current_price.invoke({"ticker": "THIS_IS_INVALID_TICKER"})
print(result5)
print()

## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**TODO: 생성한 모든 도구를 리스트로 정리하세요**

In [ ]:
# 팀에서 생성한 모든 도구를 리스트로 추가합니다.
CUSTOM_TOOLS = [
    get_current_price,
    get_historical_ohlcv,
    get_financial_metrics,
    get_company_disclosures
]

print("=" * 80)
print(f"총 {len(CUSTOM_TOOLS)}개의 커스텀 도구가 준비되었습니다.")
print("=" * 80, "\n")

# enumerate를 사용해 1번부터 순서대로 번호를 매겨 출력합니다.
for i, tool in enumerate(CUSTOM_TOOLS, 1):
    print(f"{i}. {tool.name}")
    print(f"   설명: {tool.description}")
    print()

## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)